# 🧠 DAY 2 — BASIC RAG (Engineer Mode)
### 🎯 End Goal

- Build working RAG
- Understand embeddings mathematically
- Understand retrieval behavior
- Compare retrieval quality

## What is RAG?

> Retrieval-Augmented Generation is a system that enhances LLM responses by retrieving relevant external knowledge and injecting it into the model's context before generating an answer.

#### Core Idea:

LLM alone → hallucinates

LLM + retrieved context → grounded

## RAG Architecture

```
User Query
   ↓
Embed Query (Ollama)
   ↓
Vector Similarity Search (FAISS)
   ↓
Top-K Chunks
   ↓
Groq LLM (context injected)
   ↓
Answer + Citations
```

---

## A Simple RAG pipeline:

```
1. Loads local documents (PDF)
2. Splits them into chunks
3. Creates embeddings
4. Stores them in a vector DB
5. Retrieves relevant chunks
6. Generates answers with citations
```

---

## 🔵 Step 1 — Document Loading

Load the Documents, Example: Loading pdf's using PyPDFLoader

In [1]:
from langchain_community.document_loaders import PyPDFLoader

file_path = r"C:\Users\koush\Desktop\HomeWorks\Sem-4\CS692\Paper Summaries\AccuMO Week 3 Paper 2 Summary.pdf"

loader = PyPDFLoader(file_path)
print(loader)
doc = loader.load()

In [2]:
print("Total pages loaded:", len(doc))

Total pages loaded: 1


In [3]:
print("\n--- Preview ---")
print(doc[0].page_content[:300])
print("\nMetadata:", doc[0].metadata)


--- Preview ---
AccuMO: Accuracy-Centric Multitask Offloading in Edge-Assisted 
Mobile Augmented Reality 
What is the Problem that Paper Aims to Solve?  
The paper addresses the challenge of efficient scheduling of offloading mul tiple tasks to edge 
servers in AR and MR applications to maximize the overall accurac

Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-02-22T16:10:49-05:00', 'author': 'Koushik Rama', 'moddate': '2026-02-22T16:10:49-05:00', 'source': 'C:\\Users\\koush\\Desktop\\HomeWorks\\Sem-4\\CS692\\Paper Summaries\\AccuMO Week 3 Paper 2 Summary.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


---
## 🔵 Step 2 — Chunking (Text Splitting)

> Chunking is the process of splitting your document text into smaller, overlapping pieces (“chunks”) so they can be embedded, stored, and retrieved effectively during RAG.

### ✂️ Why We Actually Do Chunking (The Real Need)

#### 1) Embeddings + Retrieval Work at the Chunk Level

Vector search doesn’t retrieve **“the whole PDF.”**
It retrieves the **most similar vectors**.

So you need your knowledge base represented as **many small vectors**, not one giant vector.

If you embed the entire document as one vector:

- Retrieval becomes too coarse (“everything matches a little”)
- You can’t pinpoint the exact section that answers the question, then its meaningless for us to implement RAG

#### 2) LLM Context Limits

Even if retrieval found the right info, you **can’t feed an entire PDF** into the model every time.

Chunking ensures you only send **top-k relevant chunks** to Groq (or any LLM).

This keeps you within context limits and reduces cost.

#### 3) Better Precision, Less Hallucination

Smaller chunks mean the model sees **only relevant evidence**, which:

- Reduces guessing
- Improves factual grounding
- Makes citations meaningful


#### 4) PDFs Are Page-Based, But Answers Are Paragraph-Based

`PyPDFLoader` gives you **page documents**.

But most answers live in a **paragraph or section**, not “page 7 as a whole.”

Chunking transforms:

> **Page text → retrievable passages**

This is what makes high-quality RAG possible.

---
## 🎯 What “Good Chunking” Means

A **good chunk** is:

- Big enough to contain complete meaning (not chopped mid-sentence)
- Small enough to be specific and retrievable
- Includes slight overlap so important context isn’t cut off


### 🔑 Key Parameters

**`chunk_size`**
How many characters (roughly) per chunk.

**`chunk_overlap`**
How much text repeats between consecutive chunks.


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap= 150
)
chunks = splitter.split_documents(doc)

In [5]:
print("#Chunks:", len(chunks))
print("------Chunk 0 Preview:------\n", chunks[0].page_content)

#Chunks: 8
------Chunk 0 Preview:------
 AccuMO: Accuracy-Centric Multitask Offloading in Edge-Assisted 
Mobile Augmented Reality 
What is the Problem that Paper Aims to Solve?  
The paper addresses the challenge of efficient scheduling of offloading mul tiple tasks to edge 
servers in AR and MR applications to maximize the overall accuracy. Conventional AR apps must 
execute several tasks, such as depth estimatio n and odometry, on every frame within the frame


## 🔵 Step 3 — Embeddings (Ollama `nomic-embed-text`)

> **Embeddings** are numeric vector representations of text. They convert Document chunks and User queries into vectors so we can perform **semantic search (meaning-based search)** instead of simple keyword matching.

### ❓ Why We Need Embeddings in RAG

RAG retrieval works like this:

1. **Embed every chunk** → store vectors in FAISS
2. **Embed the user query** → query vector
3. **Find the closest chunk vectors (top-k)**
4. **Send those chunks to Groq** as context

This is what enables the system to find information by **meaning**, not just matching words.

---

### What Happens When a Chunk Is Embedded?

Let’s say chunk 0 contains:

> "AccuMO is a framework that optimizes per-frame accuracy in AR systems..."

Internally:

1. Text → tokenized
2. Tokens → passed through transformer layers
3. Final hidden state → pooled into single vector
4. Output → 768-dimensional float vector

Important:

* Every chunk → fixed-length vector
* Length does NOT depend on text size
* Same embedding model → same vector dimension

So:

| Text       | Output         |
| ---------- | -------------- |
| 1 sentence | 768-dim vector |
| 1000 chars | 768-dim vector |


> ``` "What is AccuMO?" -> [0.0123, -0.9921, 0.3411, ..., 0.1187]   (768 numbers)```

> ``` "AccuMO is a framework that optimizes per-frame accuracy in AR systems..." -> [0.1422, -0.9221, 0.3812, ..., 0.1677]   (768 numbers) ```

---

### Where Do These Vectors Live?

Imagine 768-dimensional space.

Each chunk becomes a point in that space.

Example:
```
Chunk A → Vector A
Chunk B → Vector B
Chunk C → Vector C
```
If two chunks are semantically similar:

They will be close together in that space.

---

### How Similarity Is Identified

When user asks:

> "How does AccuMO optimize AR?"

We:

1. Embed the query → Query vector Q
2. Compute similarity between Q and every stored chunk vector

We'll do this in the next step

In [6]:
from langchain_ollama import OllamaEmbeddings

# Step 3: embeddings
emb = OllamaEmbeddings(model="nomic-embed-text")

In [7]:
# 1) sanity check on a query
q_vec = emb.embed_query("What is AccuMO about?")
print("Query embedding dim:", len(q_vec))
print("Query embedding first5:", q_vec[:5])

# 2) sanity check on chunks
sample_texts = [c.page_content for c in chunks]
c_vecs = emb.embed_documents(sample_texts)

print("\nNum chunk vectors:", len(c_vecs))
print("Chunk[0] embedding dim:", len(c_vecs[0]))
print("Chunk[0] first5:", c_vecs[0][:5])

Query embedding dim: 768
Query embedding first5: [-0.008334121, 0.010176035, -0.14090891, 0.02498016, 0.05131311]

Num chunk vectors: 8
Chunk[0] embedding dim: 768
Chunk[0] first5: [0.019541057, 0.08354413, -0.19436204, -0.037406318, 0.05676784]


## 🔵 Step 4 — Vector Store (FAISS) + Similarity Retrieval


A **vector store** is a database that stores:

- Each chunk’s embedding vector
- The chunk’s text
- The chunk’s metadata (source/page)

It enables **nearest-neighbor search**:

> “Find the chunks whose vectors are closest to the query vector.”

---

### 2) ❓ Why Step 4 Is Necessary

Once chunks are embedded, you need a structure that can:

- Search through **N vectors quickly** (N can be 10k+ later)
- Return the **top-k most similar chunks**
- Preserve metadata for **citations**

**FAISS** is used because it is:

- ⚡ Very fast
- 🖥️ Local (no external service needed)
- 📈 Scales well to large datasets

---

### 3) How Similarity Is Computed (What FAISS Is Doing)

You will:

1. Embed query → vector **q**
2. Compare **q** vs every chunk vector **vᵢ**
3. Score similarity using a distance function

Common metrics:

- **Cosine similarity** (direction-based)
- **L2 distance** (Euclidean distance)

FAISS returns the chunks with the **best scores**.

**Intuition:**
> “Close vectors” ≈ “Similar meaning”

---

### 4) How Similarity Is Identified in background

When user asks:

> *"How does AccuMO optimize AR?"*

We:

1. Embed the query → Query vector **Q**
2. Compute similarity between **Q** and every stored chunk vector

---

### 5) The Math — Cosine Similarity

The similarity is calculated as:

$$
\text{similarity}(A,B) = \frac{A \cdot B}{\|A\| \|B\|}
$$


Where:

$$A \cdot B\ = dot \ product $$
$$\|A\|\ = magnitude \ of \ vector \ A $$


**Range:** −1 to 1

| Score | Meaning |
|------|---------|
| 1 | identical direction |
| 0.8 | very similar |
| 0 | unrelated |
| −1 | opposite meaning |

FAISS computes this **very efficiently**.

---

### 6) What Actually Happens During Retrieval

When you call:

```python
retriever.invoke(query)
```
Internally:

1. Query → embedding Q
2. Compute similarity(Q, chunkᵢ) for all chunks
3. Sort by similarity (descending)
4. Return top-k chunks
---
### 7) Why Chunk Size Matters Here

If chunk is **too large**:

- Vector becomes “averaged meaning”
- Precision drops

If chunk is **too small**:

- Meaning becomes fragmented
- Important context lost

✅ Chunking directly affects **vector quality**.

---

### 8) What FAISS Stores

FAISS stores a matrix:

(num_chunks, embedding_dim)


**Example:**


(125 chunks, 768 dimensions)


- Each row = one chunk vector
- Metadata is stored alongside in the LangChain wrapper


In [8]:
from langchain_community.vectorstores import FAISS

# Step 4: Build FAISS index
db = FAISS.from_documents(chunks, emb)
retriever = db.as_retriever(search_kwargs={"k": 3})

# Step 4: Retrieval sanity test (NO Groq yet)
query = "What is AccuMO and what problem does it solve?"
results = retriever.invoke(query)

print("\nTop results:", len(results))
for i, d in enumerate(results, 1):
    src = d.metadata.get("source")
    page = d.metadata.get("page")
    print(f"\n--- Result {i} ---")
    print("Source:", src)
    print("Page:", page)
    print(d.page_content[:350])


Top results: 3

--- Result 1 ---
Source: C:\Users\koush\Desktop\HomeWorks\Sem-4\CS692\Paper Summaries\AccuMO Week 3 Paper 2 Summary.pdf
Page: 0
critical tasks in mobile AR environments. 
What are the Limitations of the Proposed Work?  
Although AccuMO is very effective it does have its own limitations. It is evaluated using synthetic 
CARLA dataset, which may not represent the variability of real world. The framework is based on 
only two tasks (odometry and depth estimation) and requires 

--- Result 2 ---
Source: C:\Users\koush\Desktop\HomeWorks\Sem-4\CS692\Paper Summaries\AccuMO Week 3 Paper 2 Summary.pdf
Page: 0
AccuMO: Accuracy-Centric Multitask Offloading in Edge-Assisted 
Mobile Augmented Reality 
What is the Problem that Paper Aims to Solve?  
The paper addresses the challenge of efficient scheduling of offloading mul tiple tasks to edge 
servers in AR and MR applications to maximize the overall accuracy. Conventional AR apps must 
execute several task

--- Result 3 ---
Source

### 1️⃣ Import FAISS Wrapper
```
from langchain_community.vectorstores import FAISS
```
This is NOT the raw FAISS C++ library.

This is:

LangChain wrapper

That connects:

* your chunks
* your embedding model
* the FAISS index

It handles:

* embedding storage
* metadata storage
* similarity search

---
### 2️⃣ Build the Vector Index
```
db = FAISS.from_documents(chunks, emb)
```
This is the most important line.

What happens internally:

For each chunk:

1. Extract chunk.page_content
2. Call:
```emb.embed_documents(...)```

3. Generate vector (768-dim)
4. Store vector in FAISS index
5. Store metadata alongside it

Conceptually:

If you had:

20 chunks

Then FAISS now stores:

Matrix shape = (20, 768)

Each row = vector representation of a chunk.

And internally it stores:

```
vector_i  →  chunk_i text + metadata
```
So it can return both.

---
### 3️⃣ Create Retriever Object
```
retriever = db.as_retriever(search_kwargs={"k": 3})
```

This converts the vector store into a retrieval interface.

Meaning:

When you give it a query, it:

1. Embeds query
2. Computes similarity with all vectors
3. Returns top k=3 chunks
    ```
    What does "k": 3 mean?
    Return the top 3 most similar chunks.
    If:
    * k = 1 → very strict, might miss context
    * k = 5 → more context, more noise
    * k too high → LLM sees irrelevant info
    ```
3 is a good starting point.

---
### 4️⃣ Query
```
query = "What is AccuMO and what problem does it solve?"
```
This is raw natural language.

Nothing special yet.

---
### 5️⃣ Retrieval Call
```
results = retriever.invoke(query)
```
This is the key step.

What happens internally:

1. emb.embed_query(query)
2. Get query vector Q
3. Compute similarity:
    similarity(Q, chunk_i_vector)
4. Sort by highest similarity
5. Return top 3 chunks

---
---
## 🔵 Step 5 — Groq LLM Integration

> This step connects your system to a text generation model (Groq).
So far, we’ve built the retrieval side (chunks → embeddings → FAISS → top-k results).
Now we verify the generation side works independently before we combine them in Step 6 (full RAG).

In [9]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [10]:
load_dotenv()

True

In [11]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0 #Be less creative
)

resp =llm.invoke("Say 'groq is connected' and nothing else")
print(resp.content)

Groq is connected.


---
## 🔵 Step 6 — Full RAG (Retriever → Context → Groq → Answer + Citations)

#### What this step is

We will combine:

- **Retriever (FAISS)** → finds top-k chunks
- **Groq LLM** → generates the answer using those chunks as evidence
- **Citations** → show source + page for each chunk used

#### Why it’s needed

Retrieval alone gives you raw text.
LLM alone may hallucinate.
RAG = retrieval supplies evidence + LLM explains it.

# LET'S PUT ALL TOGETHER

In [12]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

In [13]:
load_dotenv()

True

In [14]:
# ---- CONFIG -------
PDF_PATH = r"C:\Users\koush\Desktop\HomeWorks\Sem-4\CS692\Paper Summaries\AccuMO Week 3 Paper 2 Summary.pdf"
TOP_K = 3
# -----------

In [15]:
# Step:1 Load File
docs = PyPDFLoader(PDF_PATH).load()

In [16]:
# Step:2 Chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap= 150
)
chunks = splitter.split_documents(docs)

In [17]:
# Step:3 Embeddings
emb = OllamaEmbeddings(model="nomic-embed-text")

In [18]:
# Step:4 Vector DB (FAISS)
db = FAISS.from_documents(chunks,emb)
retriever = db.as_retriever(search_kwargs={"k": 3})

In [19]:
# Step:5 LLM setting
llm = ChatGroq(model="llama-3.1-8b-instant",temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Answer ONLY using the provided context. "
     "If the answer is not in the context, say: 'Not found in the document.'"),
    ("human",
     "Question: {question}\n\n"
     "Context:\n{context}\n\n"
     "Return a concise answer.")
])

In [20]:
# Step:6 formatting
def format_context(docs):
    blocks=[]
    for i,d in enumerate(docs, 1):
        src = d.metadata.get("source","unknown")
        page = d.metadata.get("page","n/a")
        blocks.append(f"[{i}](source={src},page={page})\n{d.page_content})")
    return "\n\n".join(blocks)

def format_citations(docs):
    cites = []
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", "n/a")
        cites.append(f"[{i}] {src} — page {page}")
    return "\n".join(cites)

In [21]:
question = "What is AccuMO and what problem does it solve?"

In [22]:
retrieved = retriever.invoke(question)
context = format_context(retrieved)

In [23]:
formatted_prompt = prompt.format_prompt(question=question, context=context)
response = llm.invoke(formatted_prompt).content

In [24]:
print("\n=== ANSWER ===")
print(response)

print("\n=== CITATIONS ===")
print(format_citations(retrieved))


=== ANSWER ===
AccuMO is an accuracy-centric multitask offloading framework in edge-assisted mobile augmented reality. It solves the problem of efficient scheduling of offloading multiple tasks to edge servers in AR and MR applications to maximize overall accuracy.

=== CITATIONS ===
[1] C:\Users\koush\Desktop\HomeWorks\Sem-4\CS692\Paper Summaries\AccuMO Week 3 Paper 2 Summary.pdf — page 0
[2] C:\Users\koush\Desktop\HomeWorks\Sem-4\CS692\Paper Summaries\AccuMO Week 3 Paper 2 Summary.pdf — page 0
[3] C:\Users\koush\Desktop\HomeWorks\Sem-4\CS692\Paper Summaries\AccuMO Week 3 Paper 2 Summary.pdf — page 0


### The RAG System is now successfully working.

We've successfully implemented the pipeline.

```
1. Loads local documents (PDF)
2. Splits them into chunks
3. Creates embeddings
4. Stores them in a vector DB
5. Retrieves relevant chunks
6. Generates answers with citations
```
